# Q-learning for execution

Adnan Masood · University of South Florida

Practice a schedule for buying ten abstract inventory units over ten steps. At each step the simulated agent may buy zero, one, or two units. These units are not specified shares, and the notebook does not place trades.

**Dataset:** The computer runs 2,000 simulated practice episodes. Seed 5 fixes the random price changes so the exercise can be repeated. Each episode has ten steps, starts with ten units to buy, and charges more impact for larger orders plus a penalty for units left at the end.

**Task:** Q-learning stores a score for each time and remaining-inventory situation, then improves those scores from simulated costs. Follow the best-scoring action at each step and report all ten actions, units left, and the deterministic evaluation cost. The final penalty makes unfinished inventory count as a cost.

**Compare:** Compare the learned schedule with TWAP (time-weighted average price), a simple rule that buys one unit in each of ten steps. Both start and finish with the same ten units. The evaluation omits random price noise and compares modeled impact plus any unfinished-inventory penalty; it does not prove which policy will trade better in a changing market.

**Runtime:** NumPy runs the simulator. Training uses random price changes; deterministic evaluation removes that noise so the policies can be compared on one fixed path. The same impact and unfinished-inventory penalty are applied to the learned policy and TWAP. A zero-noise evaluation is not a test of performance across real market conditions.


In [ ]:
import numpy as np
rng = np.random.default_rng(5)
T,inventory0,actions = 10,10,np.array([0,1,2])
Q = np.zeros((T+1,inventory0+1,len(actions)))
alpha,gamma,eps = .15,.95,.20
impact_coefficient = .08
incomplete_penalty = 5.0
price_noise_sd = .10
for episode in range(2000):
    inv = inventory0
    for t in range(T):
        valid = np.where(actions<=inv)[0]
        a_idx = rng.choice(valid) if rng.random()<eps else valid[np.argmax(Q[t,inv,valid])]
        qty = actions[a_idx]
        impact = impact_coefficient*qty**2
        price_risk = rng.normal(0,price_noise_sd)*inv
        next_inv = inv-qty
        reward = -(impact+price_risk)
        if t==T-1: reward -= incomplete_penalty*next_inv
        next_valid = np.where(actions<=next_inv)[0]
        future = 0 if t==T-1 else Q[t+1,next_inv,next_valid].max()
        Q[t,inv,a_idx] += alpha*(reward+gamma*future-Q[t,inv,a_idx])
        inv = next_inv
policy,inv = [],inventory0
for t in range(T):
    valid = np.where(actions<=inv)[0]
    qty = actions[valid[np.argmax(Q[t,inv,valid])]]
    policy.append(int(qty));inv -= qty
cost = sum(impact_coefficient*q*q for q in policy)+incomplete_penalty*inv
print("Illustrative actions:",policy,"remaining:",int(inv))
print("Deterministic evaluation cost:",cost)
print("TWAP baseline cost:",T*impact_coefficient)
print("Reward:",-cost)


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Increase training price noise

Retrain and compare the schedule. Deterministic evaluation still reports impact plus incompletion cost, with no random-price term.

Replace:
```python
price_noise_sd = .10
```
With:
```python
price_noise_sd = .30
```

### Increase incompletion penalty

The new penalty is used in both training and deterministic evaluation; inspect remaining inventory.

Replace:
```python
incomplete_penalty = 5.0
```
With:
```python
incomplete_penalty = 10.0
```

### Increase impact coefficient

Training, deterministic evaluation, and the TWAP baseline all use the new coefficient.

Replace:
```python
impact_coefficient = .08
```
With:
```python
impact_coefficient = .20
```



In [ ]:
# Increase incomplete_penalty from 5.0 to 10.0, retrain, and record the actions.
# The changed coefficient also applies to deterministic evaluation and the same-unit TWAP comparison.


## Break it

Increase price_noise_sd from 0.10 to 0.30 and retrain. Compare the learned schedule and remaining inventory, but do not claim the policy minimizes variance: the reward still has no explicit variance penalty.


In [ ]:
# Record actions, remaining inventory, and deterministic cost at both noise levels.
# Explain which part of the training reward is absent from deterministic evaluation.


## What to submit

- The ten actions, remaining inventory, deterministic cost, and TWAP cost.
- A penalty, impact, or training-noise experiment with its changed assumptions.
- An explanation of the simulator's units, reward terms, and limits of the evaluation.

## Optional extension

The simulation adds zero-mean price noise during learning, but its reward does not directly penalize variation in outcomes. To study risk, evaluate both schedules over many different price paths and report bad-tail outcomes as well as the average; this notebook does not do that comparison.
